In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics import YOLO
from pathlib import Path
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.utils.loss import v8DetectionLoss

BASE_DIR = Path.cwd()
RUNS_DIR = BASE_DIR.parent / "runs" / "detect"
PRETRAINED_MODEL = BASE_DIR.parent / "models" / "pretrained" / "yolo11m.pt"
MODEL_CONFIG = BASE_DIR.parent / "datasets" / "pcb-defects-final" / "custom_yolov11.yaml"
DATA_CONFIG = BASE_DIR.parent / "datasets" / "pcb-defects-final" / "data.yaml"

class FocalLoss(nn.Module):
    def __init__(self, gamma=1.5, alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, pred, target):
        bce = F.binary_cross_entropy_with_logits(pred, target, reduction="none")
        p = torch.sigmoid(pred)
        p_t = p * target + (1 - p) * (1 - target)
        alpha_t = self.alpha * target + (1 - self.alpha) * (1 - target)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        return (focal_weight * bce).mean()

class FocalDetectionLoss(v8DetectionLoss):
    def __init__(self, model, gamma=1.5, alpha=0.25):
        super().__init__(model)
        self.bce = FocalLoss(gamma=gamma, alpha=alpha)

class CustomDetectionTrainer(DetectionTrainer):
    def init_criterion(self):
        return FocalDetectionLoss(self.model, gamma=1.5, alpha=0.25)

model = YOLO(str(MODEL_CONFIG))

model.load(PRETRAINED_MODEL)

for m in model.model.modules():
    if type(m).__name__ == 'Detect':
        for i in range(len(m.cv3)):
            sequential_layers = m.cv3[i]
            modified_layers = nn.Sequential(
                sequential_layers[0],
                sequential_layers[1],
                nn.Dropout(p=0.15),
                sequential_layers[2]
            )
            m.cv3[i] = modified_layers

model.train(
    trainer=CustomDetectionTrainer,
    data=str(DATA_CONFIG),
    project=str(RUNS_DIR),
    epochs=100,
    imgsz=640,
    batch=10,
    device=0,
    optimizer="AdamW",
    lr0=1e-4,
    lrf=0.01,
    warmup_epochs=5.0,
    momentum=0.9,
    bgr=0.999,
    weight_decay=5e-4,
    amp=True,
    cls=0.5,
    box=7.5,
    dfl=1.0,
    mosaic=0.8,
    mixup=0.15,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.0,
    fliplr=0.5,
    degrees=10.0,
    iou=0.5,
    conf=0.25,
    name='pcb_defect_detector_v13_mod'
)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

BASE_DIR = Path.cwd()
RUNS_DIR = BASE_DIR.parent / "runs" / "detect"
PRETRAINED_MODEL = BASE_DIR.parent / "models" / "pretrained" / "yolo11m.pt"
MODEL_CONFIG = BASE_DIR.parent / "datasets" / "pcb-defects-final" / "custom_yolov11.yaml"
DATA_CONFIG = BASE_DIR.parent / "datasets" / "pcb-defects-final" / "data.yaml"

model = YOLO(str(MODEL_CONFIG))
model.load(str(PRETRAINED_MODEL))

model.train(
    data=str(DATA_CONFIG),
    project=str(RUNS_DIR),
    epochs=30,
    freeze=2,
    imgsz=640,
    batch=8,
    device=0,
    optimizer="AdamW",
    lr0=1e-3,
    lrf=0.1,
    warmup_epochs=5.0,
    weight_decay=5e-4,
    amp=True,
    mosaic=0.0,
    mixup=0.0,
    scale=0.3,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    degrees=0.0,
    cls=0.5,
    box=7.5,
    dfl=1.0,
    name='stage1_backbone',
)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics import YOLO
from pathlib import Path
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.utils.loss import v8DetectionLoss

BASE_DIR = Path.cwd()
RUNS_DIR = BASE_DIR.parent / "runs" / "detect"
MODEL_CONFIG = BASE_DIR.parent / "datasets" / "pcb-defects-final" / "custom_yolov11.yaml"
DATA_CONFIG = BASE_DIR.parent / "datasets" / "pcb-defects-final" / "data.yaml"

class FocalLoss(nn.Module):
    def __init__(self, gamma=1.5, alpha=0.25, eps=1e-6):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.eps = eps

    def forward(self, pred, target):
        pred = torch.clamp(pred, -50, 50)
        
        bce = F.binary_cross_entropy_with_logits(pred, target, reduction="none")
        p = torch.sigmoid(pred)
        p_t = p * target + (1 - p) * (1 - target)
        p_t = torch.clamp(p_t, self.eps, 1 - self.eps)
        alpha_t = self.alpha * target + (1 - self.alpha) * (1 - target)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        loss = (focal_weight * bce).mean()
        
        if torch.isnan(loss) or torch.isinf(loss):
            return torch.tensor(0.0, requires_grad=True, device=pred.device)
        
        return loss

class FocalDetectionLoss(v8DetectionLoss):
    def __init__(self, model, gamma=1.5, alpha=0.25):
        super().__init__(model)
        self.bce = FocalLoss(gamma=gamma, alpha=alpha)

class CustomDetectionTrainer(DetectionTrainer):
    def init_criterion(self):
        return FocalDetectionLoss(self.model, gamma=1.5, alpha=0.25)

best_weights = RUNS_DIR / 'stage1_backbone' / 'weights' / 'best.pt'
model2 = YOLO(str(MODEL_CONFIG))
model2.load(str(best_weights))

for m in model2.model.modules():
    if type(m).__name__ == 'Detect':
        for i in range(len(m.cv3)):
            sequential_layers = m.cv3[i]
            modified_layers = nn.Sequential(
                sequential_layers[0],
                sequential_layers[1],
                nn.Dropout(p=0.05),
                sequential_layers[2]
            )
            m.cv3[i] = modified_layers

model2.train(
    trainer=CustomDetectionTrainer,
    data=str(DATA_CONFIG),
    project=str(RUNS_DIR),
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    optimizer="AdamW",
    lr0=1e-4,
    lrf=0.001,
    warmup_epochs=3.0,
    momentum=0.9,
    weight_decay=5e-4,
    amp=True,
    mosaic=0.8,
    mixup=0.15,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    flipud=0.0,
    degrees=10.0,
    bgr=0.999,
    cls=0.5,
    box=7.5,
    dfl=1.0,
    iou=0.5,
    conf=0.25,
    name='stage2_p2_focal',
)